In [1]:
import keras
from keras import layers
from keras import ops
import numpy as np
import tensorflow as tf

## Loading the dataset

In [2]:
batch_size = 32

In [5]:
train_ds = keras.utils.text_dataset_from_directory(
    'aclImdb/train/',
    validation_split=0.2,
    subset='training',
    seed=123,
    batch_size=batch_size
)


val_ds = keras.utils.text_dataset_from_directory(
    'aclImdb/train/',
    validation_split=0.2,
    subset="validation",
    seed=123,
    batch_size=batch_size
)

test_ds = keras.utils.text_dataset_from_directory( 
    'aclImdb/test/', 
    batch_size=batch_size)

Found 25000 files belonging to 2 classes.
Using 20000 files for training.
Found 25000 files belonging to 2 classes.
Using 5000 files for validation.
Found 25000 files belonging to 2 classes.


## Preprocess the dataset

Preprocess the original **string-valued samples** into **integer-valued token sequences** of fixed length.

In [6]:
# Max number of tokens in vocabulary
vocab_size = 10000
# Max length of sequences
sequence_length = 250

In [7]:
vectorization_layer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length
)

train_texts = train_ds.map(lambda x, y: x)
vectorization_layer.adapt(train_texts)

train_ds_int = train_ds.map(lambda x, y: (vectorization_layer(x), y)) 
val_ds_int = val_ds.map(lambda x, y: (vectorization_layer(x), y))
test_ds_int = test_ds.map(lambda x, y: (vectorization_layer(x), y))

Now define a **custom layer** for converting the **integer lists** to **word vectors**.

This layer consist of two separate **Embedding layers**. One for encoding the tokens, and one for encoding their positions in the sample. The final embeddings are obtained by summing these two separate contributions.

We alos include the ability to include a mask for ignoring the padding zeroes in the input samples as well as a `get_config` method to allow for saving the model after training.

In [8]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)

        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim, mask_zero=True
        )
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim
        )
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.supports_masking = True

    def call(self, inputs):
        length = ops.shape(inputs)[-1]
        positions = ops.arange(0, length, 1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return ops.not_equal(inputs, 0)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "sequence_length": self.sequence_length,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config

Now you are ready to **build the Transformer model**.

In [9]:
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        if mask is not None:
            padding_mask = ops.cast(mask[:, None, :], dtype="int32")
        else:
            padding_mask = None

        attention_output = self.attention(
            query=inputs, value=inputs, key=inputs, attention_mask=padding_mask
        )
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
            }
        )
        return config

Now is the time to build the **actual classifier model**. The output from the **encoder** is flattened by a global pooling layer and then fed straight into the output layer of a single neuron with **sigmoid activation**.

In [10]:
# Dimension of word embeddings (token + pos)
embed_dim = 32
dense_dim = 64 
num_heads = 2

In [11]:
inputs = keras.Input(shape=(None,), dtype="int64")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(inputs)
x = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs=inputs, outputs=outputs)

model.compile(optimizer=keras.optimizers.RMSprop(), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 32)  │    328,000 │ input_layer[0][0] │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encoder │ (None, None, 32)  │     12,736 │ positional_embed… │
│ (TransformerEncode… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ transformer_enco… │
│ (GlobalAveragePool… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         33 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 340,769 (1.30 MB)

 Trainable params: 340,769 (1.30 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
history = model.fit(
    train_ds_int, 
    batch_size=batch_size, 
    epochs=3, 
    validation_data=(val_ds_int)
)

Epoch 1/3


c:\Users\araz3\anaconda3\envs\nlp_env\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


625/625 ━━━━━━━━━━━━━━━━━━━━ 51s 73ms/step - accuracy: 0.7731 - loss: 0.4662 - val_accuracy: 0.7442 - val_loss: 0.5410
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 64ms/step - accuracy: 0.8704 - loss: 0.3109 - val_accuracy: 0.8756 - val_loss: 0.3000
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 64ms/step - accuracy: 0.8964 - loss: 0.2617 - val_accuracy: 0.8788 - val_loss: 0.3146


In [13]:
model.evaluate(test_ds_int)

782/782 ━━━━━━━━━━━━━━━━━━━━ 43s 54ms/step - accuracy: 0.8665 - loss: 0.3329


[0.332897424697876, 0.8665199875831604]

## Text generation
In addition to simple tasks like classification, the **Transformer architecture models** can also be used for more ambitious natural language processing tasks. The include **machine translation and text generation**. Now we take a look at how **Neural Network model** can be trained to produce movie reviews from an initial prompt.

We start by creating a new **Dataset object** from data in the training directory only. This time, we wont need to the sentiment labels and remove the line break tokens in order to avoid generating those later.

In [14]:
dataset = keras.utils.text_dataset_from_directory(
    directory="aclImdb/train/", 
    label_mode=None, 
    batch_size=256)

dataset = dataset.map(lambda x: tf.strings.regex_replace(x, "<br />", ""))

Found 25003 files.


Next, convert the **strings** into **integer lists** as before. To speed up the training, we restrict the sequence lengths to be somewhat shorter than before and drop the vocabulary size to **top 5000 words**.

In [16]:
sequence_length = 100
vocab_size = 5000

text_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length,
)

text_vectorization.adapt(dataset)

Next, generate specialized targets for the purpose of **training the model**. The targets are simply the same integer sequences as the samples, but shifted one token to the right. This means that the sequence length gets reduced by one token from the original ones.

In [17]:
def prepare_lm_dataset(text_batch):
    vectorized_sequences = text_vectorization(text_batch)
    x = vectorized_sequences[:, :-1]
    y = vectorized_sequences[:, 1:]
    return x, y

lm_dataset = dataset.map(prepare_lm_dataset)

Now, for the actual **text generation**, we need to employ the **decoder** part of the Transformer. The building blocks of the decoder are very similar to those of the encoder, but the attention layer now includes an additional **causal mask** to prevent the model from attending to future tokens in the sequence during training.

*(Note that the original Transformer architecture includes two separate attention layers in the decoder; the latter one allows the decoder to attend to the output of the encoder. This is essential for tasks like machine translation, where the output sequence is in a different language than the input sequence. However, with text generation the encoder part of the architecture is not used, and we can omit this layer in the implementation.)*

In [18]:
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, latent_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.latent_dim = latent_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(latent_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        causal_mask = self.get_causal_attention_mask(inputs)

        # Combine the padding mask with the causal mask
        if mask is not None:
            padding_mask = ops.cast(mask[:, None, :], dtype="int32")
            combined_mask = ops.minimum(padding_mask, causal_mask)
        else:
            combined_mask = causal_mask

        attention_output = self.attention(
            query=inputs, value=inputs, key=inputs,
            attention_mask=combined_mask
        )
        out_1 = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(out_1)
        return self.layernorm_2(out_1 + proj_output)

    def get_causal_attention_mask(self, inputs):
        input_shape = ops.shape(inputs)
        sequence_length = input_shape[1]
        i = ops.arange(sequence_length)[:, None]
        j = ops.arange(sequence_length)
        mask = ops.cast(i >= j, dtype="int32")
        return ops.reshape(mask, (1, sequence_length, sequence_length))

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "latent_dim": self.latent_dim,
                "num_heads": self.num_heads,
            }
        )
        return config

Now is the time to build the **model** with the **Transformer decoder**. Its output is directly connected to the **output layer** with the size of the **vocabulary and softmax activation** to provide a probability distribution for **token predictions**.

*Note that we choose `sparse_categorical_crossentropy` as our loss function due to the target labels consist of integers (intead of being one-hot encoded).*

In [19]:
embed_dim = 64
latent_dim = 128
num_heads = 4

inputs = keras.Input(shape=(None,), dtype="int64")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(inputs)
x = TransformerDecoder(embed_dim, latent_dim, num_heads)(x)
outputs = layers.Dense(vocab_size, activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.compile(loss="sparse_categorical_crossentropy", 
              optimizer=keras.optimizers.Adam(learning_rate=0.0005))

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 64)  │    326,400 │ input_layer_2[0]… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, None)      │          0 │ input_layer_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_decoder │ (None, None, 64)  │     83,200 │ positional_embed… │
│ (TransformerDecode… │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, None,      │    325,000 │ transformer_deco… │
│                     │ 5000)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 734,600 (2.80 MB)

 Trainable params: 734,600 (2.80 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
model.fit(lm_dataset, epochs=10)

Epoch 1/10


c:\Users\araz3\anaconda3\envs\nlp_env\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


98/98 ━━━━━━━━━━━━━━━━━━━━ 116s 1s/step - loss: 6.9693
Epoch 2/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/step - loss: 5.7241
Epoch 3/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - loss: 5.3658
Epoch 4/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 110s 1s/step - loss: 5.1796
Epoch 5/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/step - loss: 5.0729
Epoch 6/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/step - loss: 5.0007
Epoch 7/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/step - loss: 4.9473
Epoch 8/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - loss: 4.9048
Epoch 9/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/step - loss: 4.8699
Epoch 10/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/step - loss: 4.8394


Once the **model** has been trained, experiment with the **text generation** can start from an initial seed prompt.

Instead of always predicting the token with the highest probability in the predicted distribution, we scale the distribution with a parameter referred to as **temperature**. A high values of temperature tend to flatten the probability distribution, which leads to somewhat more surprising choices for next tokens.



In [21]:
tokens_index = dict(enumerate(text_vectorization.get_vocabulary()))

def sample_next(predictions, temperature=1.0):
    predictions = np.asarray(predictions).astype("float64")
    predictions = np.log(predictions) / temperature
    exp_preds = np.exp(predictions)
    predictions = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, predictions, 1)
    return np.argmax(probas)

Finally, define an **initial prompt** and check out the quality of generated text. Unfortunately, due to the small model size, the results are fairly disappointing.

In particular the restricted vocabulary leads to a lot of unknown tokens in the **generated text**.

In [24]:
temperature = 0.75

sentence = "this movie was"
generate_length = 50
for i in range(generate_length):
    tokenized_sentence = text_vectorization([sentence])
    predictions = model(tokenized_sentence, training=False)
    last_position = np.count_nonzero(tokenized_sentence[0]) # last actual token position
    next_token = sample_next(predictions[0, last_position - 1, :], temperature=temperature)
    sampled_token = tokens_index[next_token]
    sentence += " " + sampled_token
print(sentence)

this movie was a fan of the [UNK] if you get the audience and its [UNK] of [UNK] of this film with a [UNK] my own its [UNK] or [UNK] [UNK] the movie while this is the [UNK] you just [UNK] him to [UNK] of the rest of the [UNK] [UNK] in a
